# <center> Mistral OCR

In [1]:
import os

from dotenv import load_dotenv
from mistralai import Mistral
from groq import Groq

load_dotenv()

MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

groq_client = Groq(api_key=GROQ_API_KEY)
mistral_client = Mistral(MISTRAL_API_KEY)

In [2]:
# utils.mistral.py

import base64
from mistralai.models import OCRResponse

def encode_pdf(pdf_path):
    """Encode the pdf to base64."""
    try:
        with open(pdf_path, "rb") as pdf_file:
            return base64.b64encode(pdf_file.read()).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: The file {pdf_path} was not found.")
        return None
    except Exception as e:  # Added general exception handling
        print(f"Error: {e}")
        return None
        
def replace_images_in_markdown(markdown_str: str, images_dict: dict) -> str:
    """
    Replace image placeholders in markdown with base64-encoded images.

    Args:
        markdown_str: Markdown text containing image placeholders
        images_dict: Dictionary mapping image IDs to base64 strings

    Returns:
        Markdown text with images replaced by base64 data
    """
    for img_name, base64_str in images_dict.items():
        markdown_str = markdown_str.replace(
            f"![{img_name}]({img_name})", f"![{img_name}]({base64_str})"
        )
    return markdown_str

def get_combined_markdown(ocr_response: OCRResponse) -> str:
    """
    Combine OCR text and images into a single markdown document.

    Args:
        ocr_response: Response from OCR processing containing text and images

    Returns:
        Combined markdown string with embedded images
    """
    markdowns: list[str] = []
    # Extract images from page
    for page in ocr_response.pages:
        image_data = {}
        for img in page.images:
            image_data[img.id] = img.image_base64
        # Replace image placeholders with actual images
        markdowns.append(replace_images_in_markdown(page.markdown, image_data))

    return "\n\n".join(markdowns)

In [3]:
from pydantic import BaseModel, Field
from enum import Enum

# Create custom annotation formats

class ImageType(str, Enum):
    TEXT = "text"
    TABLE = "table"
    IMAGE = "image"

class Image(BaseModel):
    image_type: ImageType = Field(..., description="The type of the image. Must be one of 'text', 'table' or 'image'.")
    transcription: str = Field(..., description="A transcription of the handwritten-text.")

In [4]:
import requests
import os
import json

from mistralai.extra import response_format_from_pydantic_model

# Get document paths
documents_dir = "./documents"

documents_paths = [
    os.path.join(documents_dir, f)
    for f in os.listdir(documents_dir)
    if os.path.isfile(os.path.join(documents_dir, f))
]

documents_concat = ""
for doc_path in documents_paths:

    # Encode pdf in base64
    base64_pdf = encode_pdf(doc_path)
    
    # Call the OCR API
    pdf_response = mistral_client.ocr.process(
        model="mistral-ocr-latest",
        document={
            "type": "document_url",
            "document_url": f"data:application/pdf;base64,{base64_pdf}"
        },
        bbox_annotation_format=response_format_from_pydantic_model(Image),
        include_image_base64=False # do not retrieve the bbox images, only their annotations
        # WARNING: document annotations has a limit of 8 pages
    )

    documents_concat += get_combined_markdown(pdf_response)+"\n\n---\n\n"

# TODO: remove ![img-....jpeg](...)

In [12]:
# Form to complete...

form = """# REGISTRO CIVIL CONSULAR 
## SECCIÓN I - NACIMIENTOS
(Declaración de datos para la inscripción)
Nota importante. Antes de cumplimentar ver Instrucciones aldorso

### DATOS DEL NACIDO/A
(1) Nombre [...]
Primer apellido [...]
(2) Segundo apellido [...]
Sexo [...]

### DATOS DEL NACIMIENTO
Hora (Formato HH;MM) [...]
(3) Día [...] Mes [...] Año [...]
(4) Lugar [...]
Inscrito con fecha [...] en el Registro local de [...]
en el Tomo [...] Página [...] Número [...]

### DATOS DEL PROGENITOR A / PADRE / MADRE (5)
Nombre [...]
Primer apellido [...]
Segundo apellido [...]
Hijo/a de [...] y de [...]
Nacido/a en [...] el día [...] de [...] de [...]
Estado civil al nacer el/la hijo/a [...]
Estado civil en el momento actual [...]
Nacionalidad al nacer el/la hijo/a [...]
Nacionalidad en el momento actual [...]
Domicilio [...]

### DATOS DEL PROGENITOR B / PADRE / MADRE (5)
Nombre [...]
Primer apellido [...]
Segundo apellido [...]
Hijo/a de [...] y de [...]
Nacido/a en [...] el día [...] de [...] de [...]
Estado civil al nacer el/la hijo/a [...]
Estado civil en el momento actual [...]
Nacionalidad al nacer el/la hijo/a [...]
Nacionalidad en el momento actual [...]
Domicilio [...]

### MATRIMONIO DE LOS PROGENITORES 
(6) [...]
Día de la celebración [...] mes [...] año [...]
Lugar de la celebración [...]
Inscrito en [...]
Documento acreditativo presentado [...]

### OBSERVACIONES (7)
[...]

### INSTRUCCIONES
Cumpliméntese a máquina o con caracteres de imprenta:
(1) Únicamente se podrán practicar inscripciones de nacimiento haciendo constar dos nombres simples (p.ej. José-Luis) o uno compuesto (María del Carmen).
(2) Las inscripciones de nacimiento habrán de practicarse haciendo constar dos apellidos, incluso aquellas inscripciones de personas extranjeras que hayan adquirido la nacionalidad española.
(3) Consignar el dato en letra.
(4) Indicar localidad, distrito, provincia y Estado.
(5) En caso de tratarse de filiación no matrimonial, el progenitor que solicite la inscripción no manifestará el nombre del otro progenitor, a no ser que la filiación ya estuviera establecida respecto de éste, en cuyo caso deberá ser acreditada documentalmente. En el supuesto de filiación desconocida se harán constar nombres de ambos progenitores a los solos efectos identificadores.
(6) Indicar "Existe" o "No existe".
(7) En caso de parto múltiple se consignará el orden de nacimiento.

NOTA: Se admite la presentación de títulos para la inscripción por terceras personas. No obstante el Registro Civil no está obligado a mantener correspondencia ni a informar a los mismos, salvo que estuvieran apoderados por escrito. En el supuesto de que se promueva la incoación de ex pediente, és te s olo puede s er i nstado por I os propios i nteresados o por A bogados o P rocuradores expresamente apoderados al efecto."""

In [16]:
from IPython.display import Markdown, display

# Use LLM from Groq to complete the form

prompt = f"""Eres un asistente de IA que completa el siguiente formulario utilizando únicamente los documentos proporcionados. 
El formulario contiene "[...]" donde debe insertarse la información.
Reemplaza cada "[...]" con el valor correcto obtenido de los documentos. 
Si la información no está disponible, reemplaza con "N/A".

**Formulario a completar:**
{form}
---

**Documentos relevantes extraídos:**
{documents_concat}
---

Instrucciones:
1. Lee cuidadosamente los documentos proporcionados.
2. Reemplaza cada "[...]" en el formulario con la información correspondiente.
3. Si la información falta, utiliza "N/A".
4. Mantén exactamente el mismo formato del formulario, solo reemplaza los marcadores de posición."""

completion = groq_client.chat.completions.create(
    model="moonshotai/kimi-k2-instruct",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

llm_response = completion.choices[0].message.content
print(llm_response)

# REGISTRO CIVIL CONSULAR 
## SECCIÓN I - NACIMIENTOS
(Declaración de datos para la inscripción)
Nota importante. Antes de cumplimentar ver Instrucciones aldorso

### DATOS DEL NACIDO/A
(1) Nombre Elisberto
Primer apellido Remón
(2) Segundo apellido Rey
Sexo masculino

### DATOS DEL NACIMIENTO
Hora (Formato HH;MM) 16:00
(3) Día 19 Mes mayo Año 1947
(4) Lugar Lamadrid 591, Lomas de Zamora, Provincia de Buenos Aires, Argentina
Inscrito con fecha 28 de mayo de 1947 en el Registro local de Lomas de Zamora
en el Tomo N/A Página N/A Número 658

### DATOS DEL PROGENITOR A / PADRE / MADRE (5)
Nombre Lasarias
Primer apellido Remón
Segundo apellido N/A
Hijo/a de Alberto Remón y de Mestra Remón
Nacido/a en N/A el día N/A de N/A de N/A
Estado civil al nacer el/la hijo/a casado
Estado civil en el momento actual N/A
Nacionalidad al nacer el/la hijo/a argentina
Nacionalidad en el momento actual N/A
Domicilio Lamadrid 591

### DATOS DEL PROGENITOR B / PADRE / MADRE (5)
Nombre Josefina
Primer apellido 

---